# 03 — Feature Engineering

This notebook develops leakage-safe features from `skill_builder_data_filtered_onehot.csv` for student proficiency modeling. Features should use only information available before the interaction being predicted.


## Imports and notebook setup


In [4]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("default")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:,.3f}".format)

print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"NumPy:  {np.__version__}")


Python: 3.13.14
pandas: 3.0.5
NumPy:  2.5.2


## Project paths and data loading

Load the filtered one-hot dataset produced in notebook 01. The path helper allows this notebook to run from either the repository root or the `notebooks/` directory.


In [5]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the nearest parent directory containing pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml.")


PROJECT_ROOT = find_project_root()
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "skill_builder_data_filtered_onehot.csv"
)
OUTPUT_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "skill_builder_data_feature_eng.csv"
)
FEATURE_DICTIONARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "data_dictionary"
    / "skill_builder_data_feature_eng_data_dictionary.csv"
)

assert DATA_PATH.is_file(), f"Processed dataset not found: {DATA_PATH}"
assert FEATURE_DICTIONARY_PATH.is_file(), (
    f"Feature dictionary not found: {FEATURE_DICTIONARY_PATH}"
)
print(f"Project root: {PROJECT_ROOT}")
print(f"Feature engineering dataset: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Eventual output: {OUTPUT_DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Feature dictionary: {FEATURE_DICTIONARY_PATH.relative_to(PROJECT_ROOT)}")

df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())


Project root: W:\Workstation ExtDrive\007 Data Science\003 Data Science Projects\2026_p019 assistments_2009_2010
Feature engineering dataset: data\processed\skill_builder_data_filtered_onehot.csv
Rows:    259,386
Columns: 149


,user_id,order_id,skill_name,row_id,assignment_id,assistment_id,problem_id,original,attempt_count,ms_first_response,tutor_mode,answer_type,sequence_id,student_class_id,position,base_sequence_id,teacher_id,school_id,hint_count,hint_total,overlap_time,template_id,first_action,bottom_hint,opportunity,correct,Skill_1,Skill_2,Skill_4,Skill_5,Skill_8,Skill_9,Skill_10,Skill_11,Skill_12,Skill_13,Skill_14,Skill_15,Skill_16,Skill_17,Skill_18,Skill_21,Skill_22,Skill_24,Skill_25,Skill_26,Skill_27,Skill_32,Skill_34,Skill_35,Skill_37,Skill_39,Skill_40,Skill_42,Skill_43,Skill_46,Skill_47,Skill_48,Skill_49,Skill_50,Skill_51,Skill_53,Skill_54,Skill_58,Skill_61,Skill_63,Skill_64,Skill_65,Skill_67,Skill_69,Skill_70,Skill_74,Skill_75,Skill_76,Skill_77,Skill_79,Skill_80,Skill_81,Skill_82,Skill_83,Skill_84,Skill_85,Skill_86,Skill_91,Skill_92,Skill_94,Skill_96,Skill_97,Skill_99,Skill_101,Skill_102,Skill_104,Skill_105,Skill_110,Skill_163,Skill_165,Skill_166,Skill_173,Skill_190,Skill_193,Skill_203,Skill_204,Skill_217,Skill_221,Skill_276,Skill_277,Skill_278,Skill_279,Skill_280,Skill_290,Skill_292,Skill_293,Skill_294,Skill_295,Skill_296,Skill_297,Skill_298,Skill_299,Skill_301,Skill_303,Skill_307,Skill_308,Skill_309,Skill_310,Skill_311,Skill_312,Skill_314,Skill_317,Skill_321,Skill_322,Skill_323,Skill_324,Skill_325,Skill_331,Skill_333,Skill_334,Skill_340,Skill_343,Skill_346,Skill_348,Skill_350,Skill_356,Skill_362,Skill_365,Skill_367,Skill_368,Skill_371,Skill_375,Skill_378
0,14,21617623,Circle Graph,3958,263599,53412,93383,1,1,26271,tutor,algebra,7118,12495,1,7118,42972,1,2,2,41131,52570,1,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,14,21617632,Circle Graph,3959,263599,53436,93407,1,1,29123,tutor,algebra,7118,12495,1,7118,42972,1,0,2,29123,52570,0,0,2,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,14,21617641,Circle Graph,3960,263599,53429,93400,1,1,13779,tutor,algebra,7118,12495,1,7118,42972,1,2,2,19905,52570,1,1,3,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,14,21617650,Circle Graph,3961,263599,53448,93419,1,1,16901,tutor,algebra,7118,12495,1,7118,42972,1,2,2,22600,52570,1,1,4,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,14,21617659,Circle Graph,3962,263599,53449,93420,1,1,11079,tutor,algebra,7118,12495,1,7118,42972,1,2,2,19704,52570,1,1,5,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Prior interaction count

For each student, `prior_interaction_count` gives the number of recorded interactions that occurred before the current row. A value of `0` identifies the student's first recorded interaction.


In [6]:
df = df.sort_values(["user_id", "order_id"], kind="stable").reset_index(drop=True)

prior_interaction_count = (
    df.groupby("user_id", sort=False)
    .cumcount()
    .rename("prior_interaction_count")
)
df = pd.concat([df, prior_interaction_count], axis=1)

display(
    df[["user_id", "order_id", "prior_interaction_count"]].head(30)
)


C:\Users\helsh\AppData\Local\Temp\ipykernel_15900\4244264541.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["prior_interaction_count"] = (


,user_id,order_id,prior_interaction_count
0,14,21617623,0
1,14,21617632,1
2,14,21617641,2
3,14,21617650,3
4,14,21617659,4
5,14,21617667,5
6,14,21617675,6
7,14,21617692,7
8,14,21617731,8
9,14,21617749,9


## Prior skill interactions

For every row, calculate each active skill's number of prior appearances in the student's history. The minimum, mean, median, and maximum summarize those counts when an interaction has multiple skill tags. `is_first_skill_interaction` is `1` when none of the active skills has appeared previously for that student.


In [ ]:
skill_columns = sorted(
    (column for column in df.columns if column.startswith("Skill_")),
    key=lambda column: int(column.removeprefix("Skill_")),
)
assert skill_columns, "No Skill_* indicator columns were found."

skill_indicators = df[skill_columns].astype("uint8")
assert (skill_indicators.sum(axis=1) >= 1).all(), (
    "Every row must have at least one active skill indicator."
)

prior_skill_counts = (
    skill_indicators.groupby(df["user_id"], sort=False).cumsum()
    - skill_indicators
)
current_skill_mask = skill_indicators.eq(1)
current_prior_skill_counts = prior_skill_counts.where(current_skill_mask)

min_prior_skill_interaction_count = (
    current_prior_skill_counts.min(axis=1).astype("int32")
)
mean_prior_skill_interaction_count = current_prior_skill_counts.mean(axis=1)
median_prior_skill_interaction_count = current_prior_skill_counts.median(axis=1)
max_prior_skill_interaction_count = (
    current_prior_skill_counts.max(axis=1).astype("int32")
)
skill_history_features = pd.DataFrame(
    {
        "min_prior_skill_interaction_count": (
            min_prior_skill_interaction_count
        ),
        "mean_prior_skill_interaction_count": (
            mean_prior_skill_interaction_count
        ),
        "median_prior_skill_interaction_count": (
            median_prior_skill_interaction_count
        ),
        "max_prior_skill_interaction_count": (
            max_prior_skill_interaction_count
        ),
        "is_first_skill_interaction": (
            max_prior_skill_interaction_count.eq(0).astype("uint8")
        ),
    },
    index=df.index,
)
df = pd.concat([df, skill_history_features], axis=1)

skill_history_columns = [
    "min_prior_skill_interaction_count",
    "mean_prior_skill_interaction_count",
    "median_prior_skill_interaction_count",
    "max_prior_skill_interaction_count",
]
assert df[skill_history_columns].notna().all().all()
assert df[skill_history_columns].ge(0).all().all()
assert (
    df["min_prior_skill_interaction_count"]
    <= df["mean_prior_skill_interaction_count"]
).all()
assert (
    df["mean_prior_skill_interaction_count"]
    <= df["max_prior_skill_interaction_count"]
).all()

display(
    df[
        [
            "user_id",
            "order_id",
            "prior_interaction_count",
            "min_prior_skill_interaction_count",
            "mean_prior_skill_interaction_count",
            "median_prior_skill_interaction_count",
            "max_prior_skill_interaction_count",
            "is_first_skill_interaction",
        ]
    ].head(30)
)

del skill_indicators, prior_skill_counts, current_skill_mask
del current_prior_skill_counts, skill_history_features
del min_prior_skill_interaction_count, mean_prior_skill_interaction_count
del median_prior_skill_interaction_count, max_prior_skill_interaction_count


## Student-level historical features

These features summarize only interactions preceding the current row. `student_prior_attempts` is the cumulative sum of prior rows' `attempt_count`; the current row's attempts are excluded. Accuracy features summarize the binary first-attempt `correct` outcome.

Recent accuracy uses all available prior outcomes up to each window size. It remains missing for a student's first interaction because no historical outcome exists; `prior_interaction_count` records the amount of supporting history.


In [ ]:
required_history_columns = {"user_id", "attempt_count", "correct"}
missing_history_columns = required_history_columns.difference(df.columns)
assert not missing_history_columns, (
    f"Missing columns required for student history: {sorted(missing_history_columns)}"
)
assert df["attempt_count"].notna().all()
assert df["attempt_count"].ge(0).all()
assert set(df["correct"].unique()).issubset({0, 1})

student_groups = df.groupby("user_id", sort=False)
student_prior_attempts = (
    student_groups["attempt_count"].cumsum() - df["attempt_count"]
).astype("int64")
student_prior_correct = (
    student_groups["correct"].cumsum() - df["correct"]
).astype("int64")
student_prior_accuracy = student_prior_correct.div(
    df["prior_interaction_count"].replace(0, np.nan)
)

prior_correct = student_groups["correct"].shift(1)
recent_accuracy_features = {}
for window in (3, 5, 10):
    recent_accuracy_features[f"student_recent_accuracy_{window}"] = (
        prior_correct.groupby(df["user_id"], sort=False).transform(
            lambda history: history.rolling(
                window=window, min_periods=1
            ).mean()
        )
    )

correct_flag = df["correct"].eq(1)
correct_run_id = (~correct_flag).groupby(df["user_id"], sort=False).cumsum()
inclusive_correct_streak = correct_flag.groupby(
    [df["user_id"], correct_run_id], sort=False
).cumsum()
student_correct_streak = (
    inclusive_correct_streak.groupby(df["user_id"], sort=False)
    .shift(1)
    .fillna(0)
    .astype("int32")
)

incorrect_flag = df["correct"].eq(0)
incorrect_run_id = (
    (~incorrect_flag).groupby(df["user_id"], sort=False).cumsum()
)
inclusive_incorrect_streak = incorrect_flag.groupby(
    [df["user_id"], incorrect_run_id], sort=False
).cumsum()
student_incorrect_streak = (
    inclusive_incorrect_streak.groupby(df["user_id"], sort=False)
    .shift(1)
    .fillna(0)
    .astype("int32")
)

student_history_features = pd.DataFrame(
    {
        "student_prior_attempts": student_prior_attempts,
        "student_prior_correct": student_prior_correct,
        "student_prior_accuracy": student_prior_accuracy,
        **recent_accuracy_features,
        "student_correct_streak": student_correct_streak,
        "student_incorrect_streak": student_incorrect_streak,
        "no_prior_history": (
            df["prior_interaction_count"].eq(0).astype("uint8")
        ),
        "insufficient_history_3": (
            df["prior_interaction_count"].lt(3).astype("uint8")
        ),
        "insufficient_history_5": (
            df["prior_interaction_count"].lt(5).astype("uint8")
        ),
        "insufficient_history_10": (
            df["prior_interaction_count"].lt(10).astype("uint8")
        ),
    },
    index=df.index,
)

accuracy_columns = [
    "student_prior_accuracy",
    "student_recent_accuracy_3",
    "student_recent_accuracy_5",
    "student_recent_accuracy_10",
]
history_indicator_columns = [
    "no_prior_history",
    "insufficient_history_3",
    "insufficient_history_5",
    "insufficient_history_10",
]
observed_accuracy_values = (
    student_history_features[accuracy_columns].stack().dropna()
)
assert observed_accuracy_values.between(0, 1).all()
first_interaction_mask = df["prior_interaction_count"].eq(0)
assert (
    student_history_features.loc[first_interaction_mask, accuracy_columns]
    .isna()
    .all()
    .all()
)
assert student_history_features[history_indicator_columns].isin({0, 1}).all().all()

student_history_features = student_history_features.fillna(
    {column: 0.0 for column in accuracy_columns}
)
df = pd.concat([df, student_history_features], axis=1)

assert df[accuracy_columns].notna().all().all()
assert df.loc[first_interaction_mask, accuracy_columns].eq(0).all().all()
assert df.loc[first_interaction_mask, "student_prior_attempts"].eq(0).all()
assert df.loc[first_interaction_mask, "student_prior_correct"].eq(0).all()
assert df.loc[first_interaction_mask, "student_correct_streak"].eq(0).all()
assert df.loc[first_interaction_mask, "student_incorrect_streak"].eq(0).all()

display(
    df[
        [
            "user_id",
            "order_id",
            "attempt_count",
            "correct",
            "prior_interaction_count",
            *student_history_features.columns,
        ]
    ].head(30)
)

del student_groups, prior_correct, recent_accuracy_features
del correct_flag, correct_run_id, inclusive_correct_streak
del incorrect_flag, incorrect_run_id, inclusive_incorrect_streak
del student_prior_attempts, student_prior_correct, student_prior_accuracy
del student_correct_streak, student_incorrect_streak
del student_history_features, first_interaction_mask
del observed_accuracy_values, history_indicator_columns


## Student-level historical help-seeking features

Measure help seeking over only the student's preceding interactions. `student_prior_hint_rate` is the proportion of prior interactions with at least one requested hint, while `student_prior_bottom_hint_rate` is the proportion with a requested bottom-out hint.

Both rates are encoded as `0` when no prior interaction exists; `no_prior_history` distinguishes this cold-start value from an observed 0% rate. Current-interaction help behavior is excluded because it would leak information closely tied to the current correctness outcome.


In [ ]:
required_help_columns = {"hint_count", "bottom_hint"}
missing_help_columns = required_help_columns.difference(df.columns)
assert not missing_help_columns, (
    f"Missing columns required for help history: {sorted(missing_help_columns)}"
)
assert df["hint_count"].notna().all()
assert df["hint_count"].ge(0).all()
assert set(df["bottom_hint"].unique()).issubset({0, 1})

hint_used = df["hint_count"].gt(0).astype("uint8")
bottom_hint_used = df["bottom_hint"].eq(1).astype("uint8")
prior_hint_interactions = (
    hint_used.groupby(df["user_id"], sort=False).cumsum() - hint_used
)
prior_bottom_hint_interactions = (
    bottom_hint_used.groupby(df["user_id"], sort=False).cumsum()
    - bottom_hint_used
)
history_denominator = df["prior_interaction_count"].replace(0, np.nan)
help_history_features = pd.DataFrame(
    {
        "student_prior_hint_rate": (
            prior_hint_interactions.div(history_denominator).fillna(0)
        ),
        "student_prior_bottom_hint_rate": (
            prior_bottom_hint_interactions.div(history_denominator).fillna(0)
        ),
    },
    index=df.index,
)
assert help_history_features.notna().all().all()
assert help_history_features.stack().between(0, 1).all()
assert (
    help_history_features.loc[df["no_prior_history"].eq(1)]
    .eq(0)
    .all()
    .all()
)
df = pd.concat([df, help_history_features], axis=1)

display(
    df[
        [
            "user_id",
            "order_id",
            "hint_count",
            "bottom_hint",
            "prior_interaction_count",
            "no_prior_history",
            *help_history_features.columns,
        ]
    ].head(30)
)

del hint_used, bottom_hint_used, history_denominator
del prior_hint_interactions, prior_bottom_hint_interactions
del help_history_features


## Student-skill historical features

Create a longitudinal event history for every student-skill pair. A multi-skill interaction contributes to the history of each active skill. For the current row, summarize the active skills' prior attempts and accuracy using their minimum, mean, median, and maximum.

Recent skill accuracy uses the most recent prior interactions involving that exact skill. When a student has no prior history for an active skill, its undefined accuracy is encoded as `0`; the prior-skill interaction-count summaries retain the corresponding history context.


In [ ]:
skill_history_base_columns = [
    "skill_prior_attempts",
    "skill_prior_accuracy",
    "skill_recent_accuracy_3",
    "skill_recent_accuracy_5",
    "skill_recent_accuracy_10",
]
skill_accuracy_base_columns = [
    "skill_prior_accuracy",
    "skill_recent_accuracy_3",
    "skill_recent_accuracy_5",
    "skill_recent_accuracy_10",
]
skill_history_event_frames = []

for skill_column in skill_columns:
    active_skill_mask = df[skill_column].eq(1)
    skill_events = df.loc[
        active_skill_mask, ["user_id", "attempt_count", "correct"]
    ].copy()
    skill_events["row_index"] = skill_events.index

    student_skill_groups = skill_events.groupby("user_id", sort=False)
    skill_events["skill_prior_interaction_count_internal"] = (
        student_skill_groups.cumcount()
    )
    skill_events["skill_prior_attempts"] = (
        student_skill_groups["attempt_count"].cumsum()
        - skill_events["attempt_count"]
    )
    skill_prior_correct = (
        student_skill_groups["correct"].cumsum()
        - skill_events["correct"]
    )
    skill_events["skill_prior_accuracy"] = skill_prior_correct.div(
        skill_events["skill_prior_interaction_count_internal"].replace(
            0, np.nan
        )
    )

    for window in (3, 5, 10):
        skill_events[f"skill_recent_accuracy_{window}"] = (
            student_skill_groups["correct"].transform(
                lambda outcomes: outcomes.shift(1).rolling(
                    window=window, min_periods=1
                ).mean()
            )
        )

    skill_history_event_frames.append(
        skill_events[
            [
                "row_index",
                "skill_prior_interaction_count_internal",
                *skill_history_base_columns,
            ]
        ]
    )

skill_history_events = pd.concat(
    skill_history_event_frames, ignore_index=True
)
skill_history_events[skill_accuracy_base_columns] = (
    skill_history_events[skill_accuracy_base_columns].fillna(0)
)

first_student_skill_mask = skill_history_events[
    "skill_prior_interaction_count_internal"
].eq(0)
assert (
    skill_history_events.loc[
        first_student_skill_mask, skill_accuracy_base_columns
    ]
    .eq(0)
    .all()
    .all()
)
assert (
    skill_history_events[skill_accuracy_base_columns]
    .stack()
    .between(0, 1)
    .all()
)

skill_history_features = skill_history_events.groupby(
    "row_index", sort=False
)[skill_history_base_columns].agg(["min", "mean", "median", "max"])
skill_history_features.columns = [
    f"{statistic}_{feature}"
    for feature, statistic in skill_history_features.columns
]
skill_history_features = skill_history_features.reindex(df.index)
assert skill_history_features.notna().all().all()
df = pd.concat([df, skill_history_features], axis=1)

display(
    df[
        [
            "user_id",
            "order_id",
            *skill_history_features.columns,
        ]
    ].head(30)
)

del skill_history_event_frames, skill_history_events
del skill_events, student_skill_groups, skill_prior_correct
del active_skill_mask, first_student_skill_mask
del skill_history_features
